# Testing classifiers for jailbreaks and prompt injections
SW7 project, group S7G5, AAU Copenhagen.

We run twelve classifiers on the same labelled prompts and measure how many attacks they catch, how many harmless prompts they block and how long they take per prompt. The numbers go into the section "Testing Existing Classifiers" of our report.

- Setup and how to add a classifier: [README.md](README.md)
- What we found so far: [RESULTS.md](RESULTS.md)
- The code: `prompt_data.py` (data), `classifiers.py` (the classifiers) and `benchmark.py` (scoring and tables)

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

from huggingface_hub import get_token

from benchmark import Benchmark, rule_counts, skip_on_error
from classifiers import (LlamaGuard3, Qwen3Guard, free_memory, hf_classifier, pick_device, rule_check,
                         tfidf_models, train_embedding_lr, train_lstm, train_sklearn)
from prompt_data import load_test_prompts, load_training_data

DEVICE, DTYPE = pick_device()
print("Hugging Face login:", "yes" if get_token() else "no, so Prompt Guard 2, Llama Guard 3 and HackAPrompt are skipped")

ModuleNotFoundError: No module named 'huggingface_hub'

## 2. Data
Test prompts: the test splits of deepset (prompt injections) and jackhhao (jailbreaks), a random sample of HackAPrompt attacks that worked (only attacks), and our own prompts from `our_prompts.csv`.

Training data for the learned checks: the train split of safe-guard, without the prompts that are near copies of a test prompt.

In [ ]:
HACKAPROMPT_SAMPLE = 200

data = load_test_prompts(HACKAPROMPT_SAMPLE)
display(data.groupby(["dataset", "label"], sort=False).size().unstack(fill_value=0)
        .rename(columns={0: "harmless", 1: "attacks"}))

safe_guard = load_training_data(data["text"])

## 3. Classifiers
Each cell runs one classifier and shows its results per dataset. `all` is deepset and jackhhao together. A classifier that can't be loaded, for example without Hugging Face access, is skipped.

### Rule-based check (ours)

In [ ]:
bench = Benchmark(data)
bench.run("Rule-based check", rule_check)

### ProtectAI DeBERTa v2
Trained on jackhhao, so its jackhhao results are not a fair test.

In [ ]:
with skip_on_error("ProtectAI"):
    bench.run("ProtectAI DeBERTa v2", hf_classifier("protectai/deberta-v3-base-prompt-injection-v2", DEVICE),
              trained_on=["jackhhao"])

### Prompt Guard 2 (86M)

In [ ]:
with skip_on_error("Prompt Guard 2"):
    bench.run("Prompt Guard 2 (86M)", hf_classifier("meta-llama/Llama-Prompt-Guard-2-86M", DEVICE))

### Qwen3Guard (0.6B)
Only "Unsafe" counts as an attack, "Controversial" counts as harmless.

In [ ]:
with skip_on_error("Qwen3Guard"):
    bench.run("Qwen3Guard (0.6B)", Qwen3Guard(DEVICE, DTYPE))
free_memory()

### Llama Guard 3 (1B)
Checks for harmful content and has no category for jailbreaks or prompt injections.

In [ ]:
with skip_on_error("Llama Guard 3"):
    bench.run("Llama Guard 3 (1B)", LlamaGuard3(DEVICE, DTYPE))
free_memory()

### TF-IDF checks (trained on safe-guard)
Word TF-IDF looks at words and word pairs, char TF-IDF at groups of 2 to 5 letters. Each is combined with logistic regression (LR), an SVM or naive Bayes.

In [ ]:
for name, model in tfidf_models().items():
    check, seconds = train_sklearn(model, safe_guard)
    bench.run(name, check, training_time=seconds)

### Embedding + LR (trained on safe-guard)
MiniLM turns each prompt into a vector of its meaning, and logistic regression learns from those vectors.

In [ ]:
with skip_on_error("Embedding + LR"):
    check, seconds = train_embedding_lr(safe_guard, DEVICE)
    bench.run("Embedding + LR (MiniLM)", check, training_time=seconds)
free_memory()

### LSTM (trained on safe-guard)
Its results change a lot with the seed (see RESULTS.md), so compare a few seeds before drawing conclusions.

In [ ]:
check, seconds = train_lstm(safe_guard, DEVICE, seed=0)
bench.run("LSTM (safe-guard)", check, training_time=seconds)

## 4. Results
`training s` is the training time in seconds, for the classifiers we trained ourselves.

In [ ]:
table = bench.table()
display(table[table["dataset"] == "all"].round(3))
display(table[table["dataset"] != "all"].round(3))

### Jailbreaks vs prompt injections (detection rate in %)

In [ ]:
bench.detection_by_dataset()

### Which rules fire
A useful rule fires on many attacks and almost no harmless prompts (sub-question 1).

In [ ]:
rule_counts(data)

### Do the classifiers miss the same attacks?

In [ ]:
bench.missed_by_all()

### Look at the mistakes
Change the classifier and the dataset (`None` for all datasets).

In [ ]:
bench.mistakes("Rule-based check", "deepset")

### Rows for the Overleaf tables

In [ ]:
bench.overleaf_rows({"TF-IDF (safe-guard)": "TF-IDF (ours)"})

## 5. Save
Writes `results/classifier_results.csv` (the table above) and `results/classifier_predictions.csv` (every prompt with every classifier's answer, score and time).

In [ ]:
bench.save("results")